# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook:** c21 — GNN Surrogate Model Training  
**Author:** Jasper Cluistra  
**Last Updated:** 2026-05-17

---

### Goal
Train a Graph Neural Network (GNN) surrogate model that predicts per-member structural utilization for a given truss geometry and cross-section assignment. The surrogate replaces computationally expensive Karamba FEA during optimization.

### Inputs
| File | Description |
|------|-------------|
| `{EDGE_CSV}.csv` | Edge features with Karamba utilization labels — one row per member per sample |
| `{NODE_CSV}.csv` | Node features: coordinates, boundary conditions, tributary loads |

### Outputs
| File | Description |
|------|-------------|
| `{prefix}_model.pt` | Trained GNN weights |
| `{prefix}_scaler.pkl` | Feature scaler fitted on training data |
| `{prefix}_metadata.json` | Hyperparameters, thresholds, evaluation metrics |

# 1. Preprocessing
Load edge and node CSVs, build PyG graph objects, split into train/val/test, and initialise the GNN architecture. Adjust `EDGE_CSV` / `NODE_CSV` to point to the dataset produced by c12_15 + Karamba.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "workflows"))
from c21_surrogate_training import run_preprocessing

# ── Parameters ───────────────────────────────────────────────────────────────
EDGE_CSV        = "v6_edge_C14_S19999_D20260516"
NODE_CSV        = "v6_node_C12_S19999_D20260516"
BIDIRECTIONAL   = False    # True=240 edges (undirected), False=120 edges (best)
BATCH_SIZE      = 64       # was 32; larger batches → more stable gradients
DATA_INSPECTION = False    # print per-sample statistics

# Model architecture
# hidden_dim=64 / 4 layers → ~1.1M params (safe with weight_decay + dropout)
# num_layers=4: truss load paths span up to 7-8 hops — 3 layers couldn't see the full graph
HIDDEN_DIM      = 64       # keep; avoids overfitting risk
NUM_LAYERS      = 4        # was 3; better global force-path propagation
DROPOUT_P       = 0.3      # keep

# ── Run ───────────────────────────────────────────────────────────────────────
pre = run_preprocessing(
    EDGE_CSV, NODE_CSV,
    bidirectional   = BIDIRECTIONAL,
    batch_size      = BATCH_SIZE,
    data_inspection = DATA_INSPECTION,
    hidden_dim      = HIDDEN_DIM,
    num_layers      = NUM_LAYERS,
    dropout_p       = DROPOUT_P,
)

# 2. Training
Run the training loop with early stopping and learning-rate scheduling. Adjust `EPOCHS`, `PATIENCE`, and `POS_WEIGHT` as needed.

In [ ]:
from c21_surrogate_training import run_training

# ── Parameters ───────────────────────────────────────────────────────────────
EPOCHS            = 200
LR                = 1e-4
PATIENCE          = 60
LR_FACTOR         = 0.5
LR_PATIENCE       = 15
LR_MIN            = 1e-6
GRAD_CLIP         = 1.0
WEIGHT_DECAY      = 1e-3
DEFAULT_THRESHOLD = 0.35
MIN_PRECISION     = 0.40
# Override auto-computed pos_weight (≈4.1 from class balance).
# Lower value → fewer false positives → better design-level discrimination.
# Trade-off: recall drops from ~88% to ~70%; precision rises from ~40% to ~55-60%.
POS_WEIGHT        = 2.5    # was auto ~4.1

# ── Run ───────────────────────────────────────────────────────────────────────
train_out = run_training(
    pre,
    epochs            = EPOCHS,
    lr                = LR,
    patience          = PATIENCE,
    lr_factor         = LR_FACTOR,
    lr_patience       = LR_PATIENCE,
    lr_min            = LR_MIN,
    grad_clip         = GRAD_CLIP,
    weight_decay      = WEIGHT_DECAY,
    default_threshold = DEFAULT_THRESHOLD,
    min_precision     = MIN_PRECISION,
    pos_weight        = POS_WEIGHT,
)

# 3. Evaluation
Evaluate the trained model on the held-out test set. Reports precision, recall, F1, ROC-AUC, and per-design feasibility accuracy.

In [ ]:
from c21_surrogate_training import run_evaluation

# ── Run ───────────────────────────────────────────────────────────────────────
eval_out = run_evaluation(train_out, pre)


# 4. Export
Save the trained model weights, feature scaler, and metadata to disk. The output prefix is used by the optimizer to load the surrogate bundle.

In [ ]:
from c21_surrogate_training import run_export

# ── Run ───────────────────────────────────────────────────────────────────────
export_out = run_export(pre, train_out, eval_out)
print("Saved to:", export_out["models_dir"])
print("Files:")
for f in export_out["all_files"]:
    print(" ", f)
